# Model performance on high-repeat diagnosis subgroup

Цель ноутбука — отдельно проверить качество sequence и tabular baselines на группе пациентов с высокой повторяемостью diagnosis codes.

High-repeat subgroup определяется как top 10% held-out prediction examples по `repeat_removed_first_last`, то есть по числу repeated diagnosis events, которые потенциально удаляются first-last compression.

Группа определяется только по истории пациента до `prediction_time`; label при выделении группы не используется.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


# Paths

AUDIT_DIR = Path("ehrshot_copy_forwarding_audit")

TABULAR_RESULTS_DIR = Path("ehrshot_baseline_results")
SEQUENCE_RESULTS_DIR = Path("ehrshot_sequence_results")
NUMERIC_RESULTS_DIR = Path("ehrshot_sequence_numeric_results")

TABULAR_PREDICTIONS_PATH = TABULAR_RESULTS_DIR / "classical_baseline_heldout_predictions.csv"

OUT_DIR = Path("ehrshot_all_models_high_repeat_comparison")
OUT_DIR.mkdir(exist_ok=True)

TASKS = ["guo_readmission", "guo_icu"]

HIGH_REPEAT_THRESHOLD_SOURCE = "held_out"  # descriptive analysis
HIGH_REPEAT_Q = 0.90

N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 42

print("AUDIT_DIR:", AUDIT_DIR.resolve())
print("TABULAR_RESULTS_DIR:", TABULAR_RESULTS_DIR.resolve())
print("SEQUENCE_RESULTS_DIR:", SEQUENCE_RESULTS_DIR.resolve())
print("NUMERIC_RESULTS_DIR:", NUMERIC_RESULTS_DIR.resolve())
print("OUT_DIR:", OUT_DIR.resolve())

AUDIT_DIR: /Users/polinakorobeinikova/Практика/ehrshot_copy_forwarding_audit
TABULAR_RESULTS_DIR: /Users/polinakorobeinikova/Практика/ehrshot_baseline_results
SEQUENCE_RESULTS_DIR: /Users/polinakorobeinikova/Практика/ehrshot_sequence_results
NUMERIC_RESULTS_DIR: /Users/polinakorobeinikova/Практика/ehrshot_sequence_numeric_results
OUT_DIR: /Users/polinakorobeinikova/Практика/ehrshot_all_models_high_repeat_comparison


In [2]:
# Load audit outputs and define high-repeat group

audit_parts = []

for task_name in TASKS:
    audit_path = AUDIT_DIR / f"task_history_copy_forwarding_audit__{task_name}.csv"
    if not audit_path.exists():
        raise FileNotFoundError(f"Not found: {audit_path}")

    part = pd.read_csv(audit_path)
    part["task"] = task_name
    audit_parts.append(part)

audit_df = pd.concat(audit_parts, ignore_index=True)

needed_audit_cols = [
    "task",
    "row_id",
    "subject_id",
    "split",
    "label",
    "repeat_removed_first_last",
    "raw_sequence_length_non_static",
    "n_diag_events_raw",
    "n_unique_diag_codes",
    "n_diag_duplicate_events_over_first",
    "n_diag_codes_persistent_365d",
    "compression_ratio_first_last",
]

missing = [c for c in needed_audit_cols if c not in audit_df.columns]
if missing:
    raise ValueError(f"Missing audit columns: {missing}")

audit_df = audit_df[needed_audit_cols].copy()

threshold_rows = []

for task_name in TASKS:
    task_audit = audit_df[audit_df["task"] == task_name].copy()

    if HIGH_REPEAT_THRESHOLD_SOURCE == "held_out":
        source_df = task_audit[task_audit["split"] == "held_out"]
    elif HIGH_REPEAT_THRESHOLD_SOURCE == "train":
        source_df = task_audit[task_audit["split"] == "train"]
    elif HIGH_REPEAT_THRESHOLD_SOURCE == "all":
        source_df = task_audit
    else:
        raise ValueError(f"Unknown HIGH_REPEAT_THRESHOLD_SOURCE: {HIGH_REPEAT_THRESHOLD_SOURCE}")

    threshold = source_df["repeat_removed_first_last"].quantile(HIGH_REPEAT_Q)

    threshold_rows.append({
        "task": task_name,
        "threshold_source": HIGH_REPEAT_THRESHOLD_SOURCE,
        "high_repeat_q": HIGH_REPEAT_Q,
        "repeat_removed_threshold": threshold,
        "n_source_examples": len(source_df),
    })

threshold_df = pd.DataFrame(threshold_rows)

audit_df = audit_df.merge(
    threshold_df[["task", "repeat_removed_threshold"]],
    on="task",
    how="left",
)

audit_df["high_repeat_group"] = (
    audit_df["repeat_removed_first_last"] >= audit_df["repeat_removed_threshold"]
)

audit_heldout = audit_df[audit_df["split"] == "held_out"].copy()

high_repeat_group_summary = (
    audit_heldout
    .groupby(["task", "high_repeat_group"])
    .agg(
        n_examples=("row_id", "size"),
        n_patients=("subject_id", "nunique"),
        n_positive=("label", "sum"),
        event_rate=("label", "mean"),
        repeat_removed_median=("repeat_removed_first_last", "median"),
        raw_len_median=("raw_sequence_length_non_static", "median"),
        persistent_365d_median=("n_diag_codes_persistent_365d", "median"),
    )
    .reset_index()
)

threshold_df.to_csv(OUT_DIR / "high_repeat_thresholds.csv", index=False)
high_repeat_group_summary.to_csv(OUT_DIR / "high_repeat_group_summary_heldout.csv", index=False)

display(threshold_df)
display(high_repeat_group_summary)

,task,threshold_source,high_repeat_q,repeat_removed_threshold,n_source_examples
0,guo_readmission,held_out,0.9,398.4,2189
1,guo_icu,held_out,0.9,370.4,2037


,task,high_repeat_group,n_examples,n_patients,n_positive,event_rate,repeat_removed_median,raw_len_median,persistent_365d_median
0,guo_icu,False,1833,1146,80,0.043644,11.0,2492.0,0.0
1,guo_icu,True,204,65,5,0.024510,693.0,24215.5,40.5
2,guo_readmission,False,1970,1182,225,0.114213,18.0,3927.0,0.0
3,guo_readmission,True,219,67,35,0.159817,716.0,25969.0,41.0


In [ ]:
# Prediction loading helpers

REQUIRED_PRED_COLS = {
    "task",
    "version",
    "model",
    "calibration",
    "row_id",
    "subject_id",
    "y_true",
    "risk",
}


def load_prediction_csvs(
    source_name: str,
    results_dir: Path | None = None,
    explicit_path: Path | None = None,
) -> pd.DataFrame:
    prediction_parts = []

    candidate_files = []

    if explicit_path is not None and explicit_path.exists():
        candidate_files.append(explicit_path)

    if results_dir is not None and results_dir.exists():
        candidate_files.extend(sorted(results_dir.glob("*heldout_predictions*.csv")))
        candidate_files.extend(sorted(results_dir.glob("*predictions*.csv")))

    # unique preserving order
    seen = set()
    unique_files = []
    for path in candidate_files:
        if path not in seen:
            unique_files.append(path)
            seen.add(path)

    if len(unique_files) == 0:
        raise FileNotFoundError(
            f"No prediction files found for {source_name}. "
            f"results_dir={results_dir}, explicit_path={explicit_path}"
        )

    for path in unique_files:
        try:
            head = pd.read_csv(path, nrows=5)
        except Exception:
            continue

        cols = set(head.columns)

        # Tabular file may not have version.
        if source_name == "tabular":
            required = REQUIRED_PRED_COLS - {"version"}
        else:
            required = REQUIRED_PRED_COLS

        if required.issubset(cols):
            print(f"Loading {source_name} predictions:", path)
            prediction_parts.append(pd.read_csv(path))

    if len(prediction_parts) == 0:
        raise FileNotFoundError(
            f"Found files for {source_name}, but none had required prediction columns."
        )

    pred_df = pd.concat(prediction_parts, ignore_index=True)

    if "version" not in pred_df.columns:
        pred_df["version"] = "tabular_all_features"

    if "feature_set" not in pred_df.columns:
        pred_df["feature_set"] = source_name

    pred_df["source"] = source_name

    pred_df["row_id"] = pred_df["row_id"].astype(int)
    pred_df["subject_id"] = pred_df["subject_id"].astype(int)
    pred_df["y_true"] = pred_df["y_true"].astype(int)
    pred_df["risk"] = pred_df["risk"].astype(float)

    # Remove duplicate rows if both aggregate and per-model files were loaded.
    pred_df = pred_df.drop_duplicates(
        subset=[
            "source",
            "task",
            "version",
            "model",
            "calibration",
            "row_id",
            "subject_id",
        ]
    ).reset_index(drop=True)

    return pred_df

In [4]:
# Load all predictions

tabular_pred_df = load_prediction_csvs(
    source_name="tabular",
    explicit_path=TABULAR_PREDICTIONS_PATH,
    results_dir=TABULAR_RESULTS_DIR,
)

sequence_pred_df = load_prediction_csvs(
    source_name="sequence",
    results_dir=SEQUENCE_RESULTS_DIR,
)

numeric_pred_df = load_prediction_csvs(
    source_name="numeric_sequence",
    results_dir=NUMERIC_RESULTS_DIR,
)

all_pred_df = pd.concat(
    [tabular_pred_df, sequence_pred_df, numeric_pred_df],
    ignore_index=True,
)

all_pred_df = all_pred_df[all_pred_df["calibration"] == "platt"].copy()

print("tabular_pred_df:", tabular_pred_df.shape)
print("sequence_pred_df:", sequence_pred_df.shape)
print("numeric_pred_df:", numeric_pred_df.shape)
print("all_pred_df platt:", all_pred_df.shape)

display(
    all_pred_df
    .groupby(["source", "task", "version", "model", "calibration"])
    .agg(
        n=("row_id", "size"),
        positives=("y_true", "sum"),
        event_rate=("y_true", "mean"),
    )
    .reset_index()
    .sort_values(["task", "source", "version", "model"])
)

Loading tabular predictions: ehrshot_baseline_results/classical_baseline_heldout_predictions.csv
Loading sequence predictions: ehrshot_sequence_results/guo_icu__compressed_condition_era__GRU_1L__heldout_predictions.csv
Loading sequence predictions: ehrshot_sequence_results/guo_icu__compressed_condition_era__GRU_2L__heldout_predictions.csv
Loading sequence predictions: ehrshot_sequence_results/guo_icu__compressed_condition_era__GRU__heldout_predictions.csv
Loading sequence predictions: ehrshot_sequence_results/guo_icu__compressed_condition_era__LSTM_1L__heldout_predictions.csv
Loading sequence predictions: ehrshot_sequence_results/guo_icu__compressed_condition_era__LSTM_2L__heldout_predictions.csv
Loading sequence predictions: ehrshot_sequence_results/guo_icu__compressed_condition_era__LSTM__heldout_predictions.csv
Loading sequence predictions: ehrshot_sequence_results/guo_icu__compressed_condition_era__RETAIN_lite__heldout_predictions.csv
Loading sequence predictions: ehrshot_sequence_

,source,task,version,model,calibration,n,positives,event_rate
0,numeric_sequence,guo_icu,compressed_condition_era,GRU_1L_numeric,platt,2037,85,0.041728
1,numeric_sequence,guo_icu,compressed_condition_era,GRU_2L_numeric,platt,2037,85,0.041728
2,numeric_sequence,guo_icu,compressed_condition_era,LSTM_1L_numeric,platt,2037,85,0.041728
3,numeric_sequence,guo_icu,compressed_condition_era,LSTM_2L_numeric,platt,2037,85,0.041728
4,numeric_sequence,guo_icu,compressed_condition_era,RETAIN_lite_numeric,platt,2037,85,0.041728
5,numeric_sequence,guo_icu,compressed_dedup,GRU_1L_numeric,platt,2037,85,0.041728
6,numeric_sequence,guo_icu,compressed_dedup,GRU_2L_numeric,platt,2037,85,0.041728
7,numeric_sequence,guo_icu,compressed_dedup,LSTM_1L_numeric,platt,2037,85,0.041728
8,numeric_sequence,guo_icu,compressed_dedup,LSTM_2L_numeric,platt,2037,85,0.041728
9,numeric_sequence,guo_icu,compressed_dedup,RETAIN_lite_numeric,platt,2037,85,0.041728


In [5]:
# Primary configs for family-level comparison

PRIMARY_CONFIGS = [
    # ----------------------------
    # guo_readmission
    # ----------------------------
    {
        "family_label": "tabular",
        "task": "guo_readmission",
        "source": "tabular",
        "version": "tabular_all_features",
        "model": "HistGradientBoosting",
        "calibration": "platt",
    },
    {
        "family_label": "sequence",
        "task": "guo_readmission",
        "source": "sequence",
        "version": "compressed_dedup",
        "model": "RETAIN_lite",
        "calibration": "platt",
    },
    {
        "family_label": "numeric_sequence",
        "task": "guo_readmission",
        "source": "numeric_sequence",
        "version": "compressed_dedup",
        "model": "RETAIN_lite_numeric",
        "calibration": "platt",
    },

    # ----------------------------
    # guo_icu
    # ----------------------------
    {
        "family_label": "tabular",
        "task": "guo_icu",
        "source": "tabular",
        "version": "tabular_all_features",
        "model": "RandomForest_balanced",
        "calibration": "platt",
    },
    {
        "family_label": "sequence",
        "task": "guo_icu",
        "source": "sequence",
        "version": "compressed_condition_era",
        "model": "LSTM_1L",
        "calibration": "platt",
    },
    {
        "family_label": "numeric_sequence",
        "task": "guo_icu",
        "source": "numeric_sequence",
        "version": "compressed_hybrid",
        "model": "GRU_2L_numeric",
        "calibration": "platt",
    },
]


def config_mask(df: pd.DataFrame, cfg: dict) -> pd.Series:
    mask = pd.Series(True, index=df.index)
    for col in ["task", "source", "version", "model", "calibration"]:
        mask &= df[col].eq(cfg[col])
    return mask


# Check configs are present.
for cfg in PRIMARY_CONFIGS:
    n_rows = int(config_mask(all_pred_df, cfg).sum())
    print(cfg["task"], cfg["family_label"], cfg["source"], cfg["version"], cfg["model"], "rows:", n_rows)

guo_readmission tabular tabular tabular_all_features HistGradientBoosting rows: 2189
guo_readmission sequence sequence compressed_dedup RETAIN_lite rows: 2189
guo_readmission numeric_sequence numeric_sequence compressed_dedup RETAIN_lite_numeric rows: 2189
guo_icu tabular tabular tabular_all_features RandomForest_balanced rows: 2037
guo_icu sequence sequence compressed_condition_era LSTM_1L rows: 2037
guo_icu numeric_sequence numeric_sequence compressed_hybrid GRU_2L_numeric rows: 2037


In [6]:
# Merge predictions with high-repeat labels

audit_small = audit_heldout[
    [
        "task",
        "row_id",
        "subject_id",
        "label",
        "high_repeat_group",
        "repeat_removed_first_last",
        "raw_sequence_length_non_static",
        "n_diag_events_raw",
        "n_unique_diag_codes",
        "n_diag_duplicate_events_over_first",
        "n_diag_codes_persistent_365d",
        "compression_ratio_first_last",
    ]
].copy()

audit_small["row_id"] = audit_small["row_id"].astype(int)
audit_small["subject_id"] = audit_small["subject_id"].astype(int)

selected_parts = []

for cfg in PRIMARY_CONFIGS:
    part = all_pred_df[config_mask(all_pred_df, cfg)].copy()
    part["family_label"] = cfg["family_label"]
    selected_parts.append(part)

selected_pred_df = pd.concat(selected_parts, ignore_index=True)

comparison_merged_df = selected_pred_df.merge(
    audit_small,
    on=["task", "row_id", "subject_id"],
    how="inner",
)

if len(comparison_merged_df) == 0:
    raise ValueError("Merge produced 0 rows. Check row_id/task/subject_id alignment.")

comparison_merged_df.to_csv(
    OUT_DIR / "all_models_high_repeat_predictions_merged.csv",
    index=False,
)

print("comparison_merged_df:", comparison_merged_df.shape)

display(
    comparison_merged_df
    .groupby(["task", "family_label", "high_repeat_group"])
    .agg(
        n=("row_id", "size"),
        positives=("y_true", "sum"),
        event_rate=("y_true", "mean"),
        n_patients=("subject_id", "nunique"),
        repeat_removed_median=("repeat_removed_first_last", "median"),
    )
    .reset_index()
)

comparison_merged_df: (12678, 21)


,task,family_label,high_repeat_group,n,positives,event_rate,n_patients,repeat_removed_median
0,guo_icu,numeric_sequence,False,1833,80,0.043644,1146,11.0
1,guo_icu,numeric_sequence,True,204,5,0.024510,65,693.0
2,guo_icu,sequence,False,1833,80,0.043644,1146,11.0
3,guo_icu,sequence,True,204,5,0.024510,65,693.0
4,guo_icu,tabular,False,1833,80,0.043644,1146,11.0
5,guo_icu,tabular,True,204,5,0.024510,65,693.0
6,guo_readmission,numeric_sequence,False,1970,225,0.114213,1182,18.0
7,guo_readmission,numeric_sequence,True,219,35,0.159817,67,716.0
8,guo_readmission,sequence,False,1970,225,0.114213,1182,18.0
9,guo_readmission,sequence,True,219,35,0.159817,67,716.0


In [ ]:
# Metric helpers

def safe_metric(y_true, y_prob, metric_name: str):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_prob = np.clip(y_prob, 1e-6, 1 - 1e-6)

    if len(y_true) == 0:
        return np.nan

    if metric_name == "auroc":
        if len(np.unique(y_true)) < 2:
            return np.nan
        return roc_auc_score(y_true, y_prob)

    if metric_name == "auprc":
        if len(np.unique(y_true)) < 2:
            return np.nan
        return average_precision_score(y_true, y_prob)

    if metric_name == "brier":
        return brier_score_loss(y_true, y_prob)

    if metric_name == "logloss":
        return log_loss(y_true, y_prob, labels=[0, 1])

    if metric_name.startswith("top_") and metric_name.endswith("_precision"):
        pct = int(metric_name.replace("top_", "").replace("pct_precision", ""))
        k = max(1, int(np.ceil(len(y_true) * pct / 100)))
        order = np.argsort(-y_prob)
        return float(y_true[order[:k]].mean())

    raise ValueError(f"Unknown metric: {metric_name}")


def compute_metrics(y_true, y_prob) -> dict:
    metrics = {}
    for metric_name in [
        "auroc",
        "auprc",
        "brier",
        "logloss",
        "top_5pct_precision",
        "top_10pct_precision",
        "top_20pct_precision",
    ]:
        metrics[metric_name] = safe_metric(y_true, y_prob, metric_name)
    return metrics

In [8]:
# Compute subgroup performance table

perf_rows = []

group_cols = [
    "task",
    "family_label",
    "source",
    "version",
    "model",
    "calibration",
    "high_repeat_group",
]

for key, g in comparison_merged_df.groupby(group_cols, dropna=False):
    task, family_label, source, version, model, calibration, high_repeat_group = key

    y = g["y_true"].to_numpy()
    p = g["risk"].to_numpy()

    metric_values = compute_metrics(y, p)

    perf_rows.append({
        "task": task,
        "family_label": family_label,
        "source": source,
        "version": version,
        "model": model,
        "calibration": calibration,
        "high_repeat_group": bool(high_repeat_group),
        "group_name": "high_repeat_top10" if high_repeat_group else "other_90",
        "n_examples": int(len(g)),
        "n_patients": int(g["subject_id"].nunique()),
        "n_positive": int(g["y_true"].sum()),
        "event_rate": float(g["y_true"].mean()),
        "repeat_removed_median": float(g["repeat_removed_first_last"].median()),
        "raw_len_median": float(g["raw_sequence_length_non_static"].median()),
        **metric_values,
    })

subgroup_perf_df = pd.DataFrame(perf_rows)

subgroup_perf_df.to_csv(
    OUT_DIR / "all_models_high_repeat_subgroup_performance.csv",
    index=False,
)

display(
    subgroup_perf_df
    .sort_values(["task", "group_name", "family_label"])
)

,task,family_label,source,version,model,calibration,high_repeat_group,group_name,n_examples,n_patients,n_positive,event_rate,repeat_removed_median,raw_len_median,auroc,auprc,brier,logloss,top_5pct_precision,top_10pct_precision,top_20pct_precision
1,guo_icu,numeric_sequence,numeric_sequence,compressed_hybrid,GRU_2L_numeric,platt,True,high_repeat_top10,204,65,5,0.024510,693.0,24215.5,0.877387,0.170348,0.024142,0.107450,0.181818,0.095238,0.097561
3,guo_icu,sequence,sequence,compressed_condition_era,LSTM_1L,platt,True,high_repeat_top10,204,65,5,0.024510,693.0,24215.5,0.755779,0.274907,0.022583,0.104408,0.090909,0.095238,0.073171
5,guo_icu,tabular,tabular,tabular_all_features,RandomForest_balanced,platt,True,high_repeat_top10,204,65,5,0.024510,693.0,24215.5,0.825126,0.108331,0.023545,0.101736,0.181818,0.095238,0.073171
0,guo_icu,numeric_sequence,numeric_sequence,compressed_hybrid,GRU_2L_numeric,platt,False,other_90,1833,1146,80,0.043644,11.0,2492.0,0.788669,0.223131,0.039555,0.163701,0.195652,0.179348,0.133515
2,guo_icu,sequence,sequence,compressed_condition_era,LSTM_1L,platt,False,other_90,1833,1146,80,0.043644,11.0,2492.0,0.720693,0.176286,0.039613,0.163550,0.228261,0.152174,0.108992
4,guo_icu,tabular,tabular,tabular_all_features,RandomForest_balanced,platt,False,other_90,1833,1146,80,0.043644,11.0,2492.0,0.792606,0.197060,0.039272,0.155643,0.206522,0.157609,0.119891
7,guo_readmission,numeric_sequence,numeric_sequence,compressed_dedup,RETAIN_lite_numeric,platt,True,high_repeat_top10,219,67,35,0.159817,716.0,25969.0,0.763509,0.443495,0.112440,0.370637,0.545455,0.545455,0.454545
9,guo_readmission,sequence,sequence,compressed_dedup,RETAIN_lite,platt,True,high_repeat_top10,219,67,35,0.159817,716.0,25969.0,0.737733,0.426302,0.117742,0.387277,0.545455,0.500000,0.386364
11,guo_readmission,tabular,tabular,tabular_all_features,HistGradientBoosting,platt,True,high_repeat_top10,219,67,35,0.159817,716.0,25969.0,0.720497,0.307795,0.128014,0.412398,0.454545,0.318182,0.295455
6,guo_readmission,numeric_sequence,numeric_sequence,compressed_dedup,RETAIN_lite_numeric,platt,False,other_90,1970,1182,225,0.114213,18.0,3927.0,0.777289,0.370005,0.085443,0.296796,0.494949,0.436548,0.324873


In [9]:
# Compact table

huly_perf = subgroup_perf_df.copy()

huly_perf["model_label"] = (
    huly_perf["family_label"]
    + ": "
    + huly_perf["version"]
    + " + "
    + huly_perf["model"]
)

huly_perf_compact = huly_perf[
    [
        "task",
        "group_name",
        "model_label",
        "n_examples",
        "n_positive",
        "event_rate",
        "auroc",
        "auprc",
        "brier",
        "logloss",
        "top_5pct_precision",
        "top_10pct_precision",
        "top_20pct_precision",
    ]
].copy()

for col in [
    "event_rate",
    "auroc",
    "auprc",
    "brier",
    "logloss",
    "top_5pct_precision",
    "top_10pct_precision",
    "top_20pct_precision",
]:
    huly_perf_compact[col] = huly_perf_compact[col].map(
        lambda x: f"{x:.3f}" if pd.notna(x) else "NA"
    )

huly_perf_compact.to_csv(
    OUT_DIR / "all_models_high_repeat_huly_performance_table.csv",
    index=False,
)

display(huly_perf_compact)

,task,group_name,model_label,n_examples,n_positive,event_rate,auroc,auprc,brier,logloss,top_5pct_precision,top_10pct_precision,top_20pct_precision
0,guo_icu,other_90,numeric_sequence: compressed_hybrid + GRU_2L_n...,1833,80,0.044,0.789,0.223,0.040,0.164,0.196,0.179,0.134
1,guo_icu,high_repeat_top10,numeric_sequence: compressed_hybrid + GRU_2L_n...,204,5,0.025,0.877,0.170,0.024,0.107,0.182,0.095,0.098
2,guo_icu,other_90,sequence: compressed_condition_era + LSTM_1L,1833,80,0.044,0.721,0.176,0.040,0.164,0.228,0.152,0.109
3,guo_icu,high_repeat_top10,sequence: compressed_condition_era + LSTM_1L,204,5,0.025,0.756,0.275,0.023,0.104,0.091,0.095,0.073
4,guo_icu,other_90,tabular: tabular_all_features + RandomForest_b...,1833,80,0.044,0.793,0.197,0.039,0.156,0.207,0.158,0.120
5,guo_icu,high_repeat_top10,tabular: tabular_all_features + RandomForest_b...,204,5,0.025,0.825,0.108,0.024,0.102,0.182,0.095,0.073
6,guo_readmission,other_90,numeric_sequence: compressed_dedup + RETAIN_li...,1970,225,0.114,0.777,0.370,0.085,0.297,0.495,0.437,0.325
7,guo_readmission,high_repeat_top10,numeric_sequence: compressed_dedup + RETAIN_li...,219,35,0.160,0.764,0.443,0.112,0.371,0.545,0.545,0.455
8,guo_readmission,other_90,sequence: compressed_dedup + RETAIN_lite,1970,225,0.114,0.787,0.380,0.087,0.298,0.535,0.421,0.317
9,guo_readmission,high_repeat_top10,sequence: compressed_dedup + RETAIN_lite,219,35,0.160,0.738,0.426,0.118,0.387,0.545,0.500,0.386


In [10]:
# Paired bootstrap helpers

METRICS_FOR_DELTA = [
    "auroc",
    "auprc",
    "brier",
    "logloss",
    "top_5pct_precision",
    "top_10pct_precision",
    "top_20pct_precision",
]


def summarize_values(values):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]

    if len(values) == 0:
        return {
            "mean": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "n_bootstrap_valid": 0,
        }

    return {
        "mean": float(np.mean(values)),
        "ci_low": float(np.quantile(values, 0.025)),
        "ci_high": float(np.quantile(values, 0.975)),
        "n_bootstrap_valid": int(len(values)),
    }


def paired_bootstrap_delta(df_a, df_b, metrics, n_bootstrap=N_BOOTSTRAP, seed=BOOTSTRAP_SEED):
    keep_cols = ["task", "row_id", "subject_id", "y_true", "risk"]

    a = df_a[keep_cols].rename(columns={"risk": "risk_a"})
    b = df_b[keep_cols].rename(columns={"risk": "risk_b"})

    merged = a.merge(
        b,
        on=["task", "row_id", "subject_id", "y_true"],
        how="inner",
    )

    if len(merged) == 0:
        raise ValueError("Paired merge produced 0 rows.")

    y = merged["y_true"].to_numpy()
    p_a = merged["risk_a"].to_numpy()
    p_b = merged["risk_b"].to_numpy()

    point = {
        metric: safe_metric(y, p_a, metric) - safe_metric(y, p_b, metric)
        for metric in metrics
    }

    rng = np.random.default_rng(seed)

    boot_values = {metric: [] for metric in metrics}

    n = len(merged)

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)

        y_b = y[idx]
        p_a_b = p_a[idx]
        p_b_b = p_b[idx]

        for metric in metrics:
            value_a = safe_metric(y_b, p_a_b, metric)
            value_b = safe_metric(y_b, p_b_b, metric)
            boot_values[metric].append(value_a - value_b)

    rows = []

    for metric in metrics:
        ci = summarize_values(boot_values[metric])
        rows.append({
            "metric": metric,
            "point_delta_a_minus_b": point[metric],
            "bootstrap_mean_delta": ci["mean"],
            "ci_low": ci["ci_low"],
            "ci_high": ci["ci_high"],
            "n_bootstrap_valid": ci["n_bootstrap_valid"],
            "n_paired_examples": int(len(merged)),
            "n_paired_positive": int(merged["y_true"].sum()),
        })

    return pd.DataFrame(rows)

In [11]:
# Build pairwise comparisons

PAIRWISE_COMPARISONS = []

for task_name in TASKS:
    task_cfgs = [cfg for cfg in PRIMARY_CONFIGS if cfg["task"] == task_name]

    by_family = {cfg["family_label"]: cfg for cfg in task_cfgs}

    # sequence - tabular
    PAIRWISE_COMPARISONS.append({
        "comparison": "sequence_minus_tabular",
        "task": task_name,
        "a": by_family["sequence"],
        "b": by_family["tabular"],
    })

    # numeric - sequence
    PAIRWISE_COMPARISONS.append({
        "comparison": "numeric_minus_sequence",
        "task": task_name,
        "a": by_family["numeric_sequence"],
        "b": by_family["sequence"],
    })

    # numeric - tabular
    PAIRWISE_COMPARISONS.append({
        "comparison": "numeric_minus_tabular",
        "task": task_name,
        "a": by_family["numeric_sequence"],
        "b": by_family["tabular"],
    })

PAIRWISE_COMPARISONS

[{'comparison': 'sequence_minus_tabular',
  'task': 'guo_readmission',
  'a': {'family_label': 'sequence',
   'task': 'guo_readmission',
   'source': 'sequence',
   'version': 'compressed_dedup',
   'model': 'RETAIN_lite',
   'calibration': 'platt'},
  'b': {'family_label': 'tabular',
   'task': 'guo_readmission',
   'source': 'tabular',
   'version': 'tabular_all_features',
   'model': 'HistGradientBoosting',
   'calibration': 'platt'}},
 {'comparison': 'numeric_minus_sequence',
  'task': 'guo_readmission',
  'a': {'family_label': 'numeric_sequence',
   'task': 'guo_readmission',
   'source': 'numeric_sequence',
   'version': 'compressed_dedup',
   'model': 'RETAIN_lite_numeric',
   'calibration': 'platt'},
  'b': {'family_label': 'sequence',
   'task': 'guo_readmission',
   'source': 'sequence',
   'version': 'compressed_dedup',
   'model': 'RETAIN_lite',
   'calibration': 'platt'}},
 {'comparison': 'numeric_minus_tabular',
  'task': 'guo_readmission',
  'a': {'family_label': 'numeri

In [12]:
# Run paired bootstrap inside high-repeat / other groups

paired_rows = []

for comp in PAIRWISE_COMPARISONS:
    for high_repeat_group in [True, False]:
        a_cfg = comp["a"]
        b_cfg = comp["b"]

        df_a = comparison_merged_df[
            (comparison_merged_df["task"] == comp["task"])
            & (comparison_merged_df["family_label"] == a_cfg["family_label"])
            & (comparison_merged_df["source"] == a_cfg["source"])
            & (comparison_merged_df["version"] == a_cfg["version"])
            & (comparison_merged_df["model"] == a_cfg["model"])
            & (comparison_merged_df["calibration"] == a_cfg["calibration"])
            & (comparison_merged_df["high_repeat_group"] == high_repeat_group)
        ].copy()

        df_b = comparison_merged_df[
            (comparison_merged_df["task"] == comp["task"])
            & (comparison_merged_df["family_label"] == b_cfg["family_label"])
            & (comparison_merged_df["source"] == b_cfg["source"])
            & (comparison_merged_df["version"] == b_cfg["version"])
            & (comparison_merged_df["model"] == b_cfg["model"])
            & (comparison_merged_df["calibration"] == b_cfg["calibration"])
            & (comparison_merged_df["high_repeat_group"] == high_repeat_group)
        ].copy()

        delta_df = paired_bootstrap_delta(
            df_a=df_a,
            df_b=df_b,
            metrics=METRICS_FOR_DELTA,
            n_bootstrap=N_BOOTSTRAP,
            seed=BOOTSTRAP_SEED,
        )

        for _, r in delta_df.iterrows():
            paired_rows.append({
                "task": comp["task"],
                "group_name": "high_repeat_top10" if high_repeat_group else "other_90",
                "high_repeat_group": high_repeat_group,
                "comparison": comp["comparison"],
                "model_a": f'{a_cfg["family_label"]}: {a_cfg["version"]} + {a_cfg["model"]}',
                "model_b": f'{b_cfg["family_label"]}: {b_cfg["version"]} + {b_cfg["model"]}',
                "metric": r["metric"],
                "point_delta_a_minus_b": r["point_delta_a_minus_b"],
                "bootstrap_mean_delta": r["bootstrap_mean_delta"],
                "ci_low": r["ci_low"],
                "ci_high": r["ci_high"],
                "n_bootstrap_valid": r["n_bootstrap_valid"],
                "n_paired_examples": r["n_paired_examples"],
                "n_paired_positive": r["n_paired_positive"],
            })

paired_delta_df = pd.DataFrame(paired_rows)

paired_delta_df.to_csv(
    OUT_DIR / "all_models_high_repeat_paired_bootstrap_delta.csv",
    index=False,
)

display(
    paired_delta_df
    .sort_values(["task", "group_name", "comparison", "metric"])
)

,task,group_name,high_repeat_group,comparison,model_a,model_b,metric,point_delta_a_minus_b,bootstrap_mean_delta,ci_low,ci_high,n_bootstrap_valid,n_paired_examples,n_paired_positive
57,guo_icu,high_repeat_top10,True,numeric_minus_sequence,numeric_sequence: compressed_hybrid + GRU_2L_n...,sequence: compressed_condition_era + LSTM_1L,auprc,-0.104560,-0.053498,-0.592540,0.412037,996,204,5
56,guo_icu,high_repeat_top10,True,numeric_minus_sequence,numeric_sequence: compressed_hybrid + GRU_2L_n...,sequence: compressed_condition_era + LSTM_1L,auroc,0.121608,0.125279,-0.098522,0.357030,996,204,5
58,guo_icu,high_repeat_top10,True,numeric_minus_sequence,numeric_sequence: compressed_hybrid + GRU_2L_n...,sequence: compressed_condition_era + LSTM_1L,brier,0.001559,0.001363,-0.005593,0.008426,1000,204,5
59,guo_icu,high_repeat_top10,True,numeric_minus_sequence,numeric_sequence: compressed_hybrid + GRU_2L_n...,sequence: compressed_condition_era + LSTM_1L,logloss,0.003043,0.002210,-0.026275,0.034647,1000,204,5
61,guo_icu,high_repeat_top10,True,numeric_minus_sequence,numeric_sequence: compressed_hybrid + GRU_2L_n...,sequence: compressed_condition_era + LSTM_1L,top_10pct_precision,0.000000,-0.000286,-0.142857,0.144048,1000,204,5
62,guo_icu,high_repeat_top10,True,numeric_minus_sequence,numeric_sequence: compressed_hybrid + GRU_2L_n...,sequence: compressed_condition_era + LSTM_1L,top_20pct_precision,0.024390,0.021585,0.000000,0.073171,1000,204,5
60,guo_icu,high_repeat_top10,True,numeric_minus_sequence,numeric_sequence: compressed_hybrid + GRU_2L_n...,sequence: compressed_condition_era + LSTM_1L,top_5pct_precision,0.090909,0.039818,-0.272727,0.363636,1000,204,5
71,guo_icu,high_repeat_top10,True,numeric_minus_tabular,numeric_sequence: compressed_hybrid + GRU_2L_n...,tabular: tabular_all_features + RandomForest_b...,auprc,0.062016,0.093012,-0.110992,0.473002,996,204,5
70,guo_icu,high_repeat_top10,True,numeric_minus_tabular,numeric_sequence: compressed_hybrid + GRU_2L_n...,tabular: tabular_all_features + RandomForest_b...,auroc,0.052261,0.056609,-0.074450,0.258132,996,204,5
72,guo_icu,high_repeat_top10,True,numeric_minus_tabular,numeric_sequence: compressed_hybrid + GRU_2L_n...,tabular: tabular_all_features + RandomForest_b...,brier,0.000597,0.000464,-0.006367,0.006169,1000,204,5


In [13]:
# Compact AUPRC delta table

paired_auprc = paired_delta_df[paired_delta_df["metric"] == "auprc"].copy()

paired_auprc["delta_auprc_with_ci"] = paired_auprc.apply(
    lambda r: (
        f'{r["point_delta_a_minus_b"]:+.3f} '
        f'[{r["ci_low"]:+.3f}, {r["ci_high"]:+.3f}]'
    ),
    axis=1,
)

paired_auprc_compact = paired_auprc[
    [
        "task",
        "group_name",
        "comparison",
        "model_a",
        "model_b",
        "n_paired_examples",
        "n_paired_positive",
        "delta_auprc_with_ci",
        "n_bootstrap_valid",
    ]
].copy()

paired_auprc_compact.to_csv(
    OUT_DIR / "all_models_high_repeat_paired_auprc_huly_table.csv",
    index=False,
)

display(paired_auprc_compact)

,task,group_name,comparison,model_a,model_b,n_paired_examples,n_paired_positive,delta_auprc_with_ci,n_bootstrap_valid
1,guo_readmission,high_repeat_top10,sequence_minus_tabular,sequence: compressed_dedup + RETAIN_lite,tabular: tabular_all_features + HistGradientBo...,219,35,"+0.119 [-0.038, +0.250]",1000
8,guo_readmission,other_90,sequence_minus_tabular,sequence: compressed_dedup + RETAIN_lite,tabular: tabular_all_features + HistGradientBo...,1970,225,"-0.007 [-0.070, +0.056]",1000
15,guo_readmission,high_repeat_top10,numeric_minus_sequence,numeric_sequence: compressed_dedup + RETAIN_li...,sequence: compressed_dedup + RETAIN_lite,219,35,"+0.017 [-0.096, +0.161]",1000
22,guo_readmission,other_90,numeric_minus_sequence,numeric_sequence: compressed_dedup + RETAIN_li...,sequence: compressed_dedup + RETAIN_lite,1970,225,"-0.010 [-0.062, +0.043]",1000
29,guo_readmission,high_repeat_top10,numeric_minus_tabular,numeric_sequence: compressed_dedup + RETAIN_li...,tabular: tabular_all_features + HistGradientBo...,219,35,"+0.136 [-0.004, +0.268]",1000
36,guo_readmission,other_90,numeric_minus_tabular,numeric_sequence: compressed_dedup + RETAIN_li...,tabular: tabular_all_features + HistGradientBo...,1970,225,"-0.017 [-0.069, +0.037]",1000
43,guo_icu,high_repeat_top10,sequence_minus_tabular,sequence: compressed_condition_era + LSTM_1L,tabular: tabular_all_features + RandomForest_b...,204,5,"+0.167 [-0.145, +0.575]",996
50,guo_icu,other_90,sequence_minus_tabular,sequence: compressed_condition_era + LSTM_1L,tabular: tabular_all_features + RandomForest_b...,1833,80,"-0.021 [-0.104, +0.053]",1000
57,guo_icu,high_repeat_top10,numeric_minus_sequence,numeric_sequence: compressed_hybrid + GRU_2L_n...,sequence: compressed_condition_era + LSTM_1L,204,5,"-0.105 [-0.593, +0.412]",996
64,guo_icu,other_90,numeric_minus_sequence,numeric_sequence: compressed_hybrid + GRU_2L_n...,sequence: compressed_condition_era + LSTM_1L,1833,80,"+0.047 [-0.017, +0.111]",1000


In [14]:
# Compact delta table for selected metrics

selected_metrics = [
    "auroc",
    "auprc",
    "brier",
    "logloss",
    "top_5pct_precision",
    "top_10pct_precision",
    "top_20pct_precision",
]

delta_compact = paired_delta_df[
    paired_delta_df["metric"].isin(selected_metrics)
].copy()

delta_compact["delta_with_ci"] = delta_compact.apply(
    lambda r: (
        f'{r["point_delta_a_minus_b"]:+.3f} '
        f'[{r["ci_low"]:+.3f}, {r["ci_high"]:+.3f}]'
    ),
    axis=1,
)

delta_compact_pivot = (
    delta_compact
    .pivot_table(
        index=[
            "task",
            "group_name",
            "comparison",
            "model_a",
            "model_b",
            "n_paired_examples",
            "n_paired_positive",
        ],
        columns="metric",
        values="delta_with_ci",
        aggfunc="first",
    )
    .reset_index()
)

delta_compact_pivot.to_csv(
    OUT_DIR / "all_models_high_repeat_paired_delta_all_metrics_huly_table.csv",
    index=False,
)

display(delta_compact_pivot)

metric,task,group_name,comparison,model_a,model_b,n_paired_examples,n_paired_positive,auprc,auroc,brier,logloss,top_10pct_precision,top_20pct_precision,top_5pct_precision
0,guo_icu,high_repeat_top10,numeric_minus_sequence,numeric_sequence: compressed_hybrid + GRU_2L_n...,sequence: compressed_condition_era + LSTM_1L,204,5,"-0.105 [-0.593, +0.412]","+0.122 [-0.099, +0.357]","+0.002 [-0.006, +0.008]","+0.003 [-0.026, +0.035]","+0.000 [-0.143, +0.144]","+0.024 [+0.000, +0.073]","+0.091 [-0.273, +0.364]"
1,guo_icu,high_repeat_top10,numeric_minus_tabular,numeric_sequence: compressed_hybrid + GRU_2L_n...,tabular: tabular_all_features + RandomForest_b...,204,5,"+0.062 [-0.111, +0.473]","+0.052 [-0.074, +0.258]","+0.001 [-0.006, +0.006]","+0.006 [-0.034, +0.034]","+0.000 [-0.143, +0.143]","+0.024 [+0.000, +0.073]","+0.000 [-0.182, +0.364]"
2,guo_icu,high_repeat_top10,sequence_minus_tabular,sequence: compressed_condition_era + LSTM_1L,tabular: tabular_all_features + RandomForest_b...,204,5,"+0.167 [-0.145, +0.575]","-0.069 [-0.347, +0.206]","-0.001 [-0.006, +0.003]","+0.003 [-0.024, +0.026]","+0.000 [-0.096, +0.096]","+0.000 [-0.049, +0.073]","-0.091 [-0.273, +0.273]"
3,guo_icu,other_90,numeric_minus_sequence,numeric_sequence: compressed_hybrid + GRU_2L_n...,sequence: compressed_condition_era + LSTM_1L,1833,80,"+0.047 [-0.017, +0.111]","+0.068 [+0.017, +0.123]","-0.000 [-0.002, +0.002]","+0.000 [-0.008, +0.010]","+0.027 [-0.027, +0.065]","+0.025 [-0.003, +0.046]","-0.033 [-0.098, +0.022]"
4,guo_icu,other_90,numeric_minus_tabular,numeric_sequence: compressed_hybrid + GRU_2L_n...,tabular: tabular_all_features + RandomForest_b...,1833,80,"+0.026 [-0.060, +0.095]","-0.004 [-0.053, +0.046]","+0.000 [-0.002, +0.003]","+0.008 [-0.003, +0.020]","+0.022 [-0.033, +0.060]","+0.014 [-0.014, +0.038]","-0.011 [-0.098, +0.065]"
5,guo_icu,other_90,sequence_minus_tabular,sequence: compressed_condition_era + LSTM_1L,tabular: tabular_all_features + RandomForest_b...,1833,80,"-0.021 [-0.104, +0.053]","-0.072 [-0.128, -0.021]","+0.000 [-0.001, +0.003]","+0.008 [-0.002, +0.020]","-0.005 [-0.049, +0.033]","-0.011 [-0.038, +0.014]","+0.022 [-0.065, +0.098]"
6,guo_readmission,high_repeat_top10,numeric_minus_sequence,numeric_sequence: compressed_dedup + RETAIN_li...,sequence: compressed_dedup + RETAIN_lite,219,35,"+0.017 [-0.096, +0.161]","+0.026 [-0.030, +0.084]","-0.005 [-0.015, +0.004]","-0.017 [-0.046, +0.013]","+0.045 [-0.182, +0.227]","+0.068 [-0.068, +0.182]","+0.000 [-0.273, +0.364]"
7,guo_readmission,high_repeat_top10,numeric_minus_tabular,numeric_sequence: compressed_dedup + RETAIN_li...,tabular: tabular_all_features + HistGradientBo...,219,35,"+0.136 [-0.004, +0.268]","+0.043 [-0.044, +0.118]","-0.016 [-0.030, -0.002]","-0.042 [-0.086, +0.000]","+0.227 [-0.045, +0.409]","+0.159 [+0.000, +0.273]","+0.091 [-0.182, +0.455]"
8,guo_readmission,high_repeat_top10,sequence_minus_tabular,sequence: compressed_dedup + RETAIN_lite,tabular: tabular_all_features + HistGradientBo...,219,35,"+0.119 [-0.038, +0.250]","+0.017 [-0.066, +0.094]","-0.010 [-0.023, +0.001]","-0.025 [-0.067, +0.013]","+0.182 [-0.045, +0.364]","+0.091 [-0.091, +0.228]","+0.091 [-0.273, +0.364]"
9,guo_readmission,other_90,numeric_minus_sequence,numeric_sequence: compressed_dedup + RETAIN_li...,sequence: compressed_dedup + RETAIN_lite,1970,225,"-0.010 [-0.062, +0.043]","-0.010 [-0.035, +0.016]","-0.001 [-0.005, +0.002]","-0.002 [-0.012, +0.008]","+0.015 [-0.046, +0.076]","+0.008 [-0.018, +0.041]","-0.040 [-0.131, +0.081]"


## Основные выводы

В этом ноутбуке сравниваются три семейства моделей на подгруппах пациентов с большим количеством повторяющихся диагнозов:

1. tabular baseline модели, использующие агрегированные признаки;
2. sequence модель, использующая только коды;
3. sequence-numeric модель, учитывающая числовые значения.

Цель анализа — проверить, особенно ли полезны последовательностные модели для пациентов, в истории которых присутствует большое количество повторяющихся диагностических событий.

Подгруппа high-repeat определялась как верхние 10% примеров из отложенной выборки по метрике `repeat_removed_first_last`. Подгруппа формируется только на основе истории пациента до `prediction_time`; целевая переменная при выделении группы не используется.

### Размер подгруппы high-repeat

| Задача            | Порог high-repeat | High-repeat n / positives | Частота события в high-repeat | Частота события в остальных 90% |
| ----------------- | ----------------- | ------------------------- | ----------------------------- | ------------------------------- |
| `guo_readmission` | 398.4             | 219 / 35                  | 0.160                         | 0.114                           |
| `guo_icu`         | 370.4             | 204 / 5                   | 0.025                         | 0.044                           |

Для `guo_readmission` пациенты из группы high-repeat имеют более высокую вероятность положительного исхода по сравнению с остальной частью отложенной выборки. Для `guo_icu` наблюдается обратная ситуация: в группе high-repeat частота события ниже, а сама подгруппа содержит всего 5 положительных примеров.

---

## guo_readmission

В подгруппе high-repeat обе sequence модели превосходят tabular baseline подход по точечной оценке AUPRC.

| Семейство моделей | Модель                                   | AUROC | AUPRC | Brier ↓ |
| ----------------- | ---------------------------------------- | ----- | ----- | ------- |
| Tabular         | `HistGradientBoosting`                   | 0.720 | 0.308 | 0.128   |
| Sequence        | `compressed_dedup + RETAIN_lite`         | 0.738 | 0.426 | 0.118   |
| Numeric sequence| `compressed_dedup + RETAIN_lite_numeric` | 0.764 | 0.443 | 0.112   |

Разности AUPRC, оцененные с помощью парного бутстрэпа внутри подгруппы high-repeat:

| Сравнение                   | Δ AUPRC | 95% CI           |
| --------------------------- | ------- | ---------------- |
| Sequence - Tabular          | +0.119  | [-0.038, +0.250] |
| Numeric sequence - Sequence | +0.017  | [-0.096, +0.161] |
| Numeric sequence - Tabular  | +0.136  | [-0.004, +0.268] |

Интерпретация результатов:

* Sequence модель только с кодами и numeric sequence модель демонстрируют положительный сигнал по AUPRC относительно tabular baseline для пациентов с большим количеством повторяющихся диагнозов.
* Numeric sequence модель имеет лучшую точечную оценку: AUPRC = 0.443.
* Однако доверительные интервалы, полученные с помощью парного бутстрэпа, всё ещё включают 0 или практически касаются его, поэтому результат следует интерпретировать как положительный, но пока не полностью убедительный сигнал.
* Numeric sequence модель также улучшает Brier score относительно табличной модели в этой подгруппе: Δ Brier = -0.016, 95% CI [-0.030, -0.002]. Поскольку меньшие значения Brier лучше, это указывает на более качественные вероятностные оценки.

В остальных 90% примеров задачи readmission различия между моделями значительно меньше:

| Семейство моделей | AUROC | AUPRC | Brier ↓ |
| ----------------- | ----- | ----- | ------- |
| Tabular         | 0.772 | 0.387 | 0.086   |
| Sequence        | 0.787 | 0.380 | 0.087   |
| Numeric sequence| 0.777 | 0.370 | 0.085   |

Все доверительные интервалы для разностей AUPRC в остальных 90% содержат 0:

| Сравнение                   | Δ AUPRC | 95% CI           |
| --------------------------- | ------- | ---------------- |
| Sequence - Tabular          | -0.007  | [-0.070, +0.056] |
| Numeric sequence - Sequence | -0.010  | [-0.062, +0.043] |
| Numeric sequence - Tabular  | -0.017  | [-0.069, +0.037] |

Таким образом, преимущество последовательностных моделей проявляется главным образом в подгруппе high-repeat для задачи readmission, а не во всей популяции пациентов.

---

## guo_icu

Для `guo_icu` подгруппа high-repeat содержит всего 5 положительных событий. Это делает оценки AUROC и AUPRC нестабильными.

Результаты для подгруппы high-repeat:

| Семейство моделей | Модель                               | AUROC | AUPRC | Brier ↓ |
| ----------------- | ------------------------------------ | ----- | ----- | ------- |
| Tabular           | `RandomForest_balanced`              | 0.825 | 0.108 | 0.024   |
| Sequence          | `compressed_condition_era + LSTM_1L` | 0.756 | 0.275 | 0.023   |
| Numeric sequence  | `compressed_hybrid + GRU_2L_numeric` | 0.877 | 0.170 | 0.024   |

Разности AUPRC, оцененные с помощью парного бутстрэпа внутри ICU high-repeat подгруппы:

| Сравнение                   | Δ AUPRC | 95% CI           |
| --------------------------- | ------- | ---------------- |
| Sequence - Tabular          | +0.167  | [-0.145, +0.575] |
| Numeric sequence - Sequence | -0.105  | [-0.593, +0.412] |
| Numeric sequence - Tabular  | +0.062  | [-0.111, +0.473] |

Все интервалы очень широкие и содержат 0. Поэтому результаты для ICU high-repeat следует рассматривать исключительно как предварительные и исследовательские.

В остальных 90% примеров ICU numeric sequence модель имеет лучшую точечную оценку AUPRC:

| Семейство моделей | AUROC | AUPRC | Brier ↓ |
| ----------------- | ----- | ----- | ------- |
| Tabular           | 0.793 | 0.197 | 0.039   |
| Sequence          | 0.721 | 0.176 | 0.040   |
| Numeric sequence  | 0.789 | 0.223 | 0.040   |

Парный бутстрэп показывает, что numeric sequence модель превосходит sequence модель по AUROC в остальных 90% ICU-примеров: Δ AUROC = +0.068, 95% CI [+0.017, +0.123].

Для AUPRC также наблюдается положительная разность в пользу numeric sequence модели, однако доверительный интервал включает 0: Δ AUPRC = +0.047, 95% CI [-0.017, +0.111].

numeric sequence модель близка к табличному базовому подходу для ICU: Δ AUPRC = +0.026, 95% CI [-0.060, +0.095], а Δ AUROC = -0.004, 95% CI [-0.053, +0.046]. Это означает, что на остальных 90% ICU-примеров различия между числовой последовательностной моделью и табличным базовым подходом нельзя считать статистически надежными.

---

## Общий вывод

Наиболее сильная поддержка основной гипотезы проекта наблюдается для пациентов задачи `guo_readmission`, относящихся к подгруппе high-repeat.

В этой подгруппе большое количество повторяющихся диагнозов связано с повышенным риском целевого события, а обе sequence модели превосходят tabular baseline по точечной оценке AUPRC. Numeric sequence модель показывает лучший результат: AUPRC = 0.443 против 0.426 у sequence модели и 0.308 у tabular модели HGB.

Однако доверительные интервалы, полученные с помощью парного бутстрэпа, всё ещё содержат 0, поэтому результат корректнее описывать как выраженный положительный сигнал, а не как статистически доказанное превосходство.

Для `guo_icu` большое количество повторяющихся диагнозов не выделяет группу пациентов с повышенным риском. Подгруппа high-repeat содержит всего 5 положительных примеров, поэтому сравнение моделей в ней нестабильно. Numeric sequence модель оказывается более полезной для ICU в более широкой группе остальных 90% пациентов, где она превосходит sequenceмодель и приближается по качеству к tabular baseline.